# 🦠 Malaria Detection Using Convolutional Neural Networks

**Author:** Fadia Baissi  
**Affiliation:** LINFI Laboratory, Mohamed Khider University of Biskra, Algeria  
**Project:** TIASM — AI-Powered Platform for E-Health  

---

## Overview

This notebook implements a **custom CNN pipeline** for binary classification of malaria-infected vs. uninfected red blood cell images.  
It covers the full pipeline as described in our published work:

```
Dataset
  └─► Data Preprocessing
        └─► Data Augmentation
              └─► CNN Initialisation
                    └─► Feature Extraction
                          └─► Training (K-Fold, K=3)
                                └─► Fine Tuning (Optuna)
                                      └─► Parasite Identification
                                            └─► Performance Metrics
```

## Dataset
- **NIH/LHNCBC Cell Image Library**: 27,558 images (public domain)  
- **Local Algerian dataset**: 442 images collected from 7 Algerian cities at NIHPT Blida  
- **Total**: ~28,000 images | **Classes**: Parasitized (1) / Uninfected (0) | **Balanced**: 14,000 per class

## Publications
- Baissi et al. (2024). *Paludism Diagnosis Using Deep Learning*. ICEIS 2024, Springer. DOI: 10.2991/978-94-6463-496-9_19  
- Baissi et al. (2024). *TIASM-platform: a New AI-Powered Platform for E-Health*. ICCSA 2024, Springer Nature.  
- Baissi & Ammari (2024). *Paludism Detection using CNN*. ISNIB 2024, IEEE. DOI: 10.1109/ISNIB64820.2025.10982943

---
**License:** MIT  
**Environment:** Python 3.9 | TensorFlow 2.10 | Keras 2.10 | Optuna 3.0

## 1. Install Dependencies

In [ ]:
# Install required packages (run once in Colab or fresh environment)
!pip install tensorflow==2.10.0 keras==2.10.0 optuna==3.0.3 optkeras==1.0.0 \
             scikit-learn==1.1.3 numpy==1.23.4 matplotlib==3.6.2 seaborn==0.12.1 -q

## 2. Imports

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential, Model
from keras.layers import (
    Conv2D, MaxPooling2D, Flatten,
    Dense, Dropout, GlobalAveragePooling2D
)
from keras.optimizers import Adam, RMSprop
from keras.callbacks import ModelCheckpoint, EarlyStopping
import keras.backend as K

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score
)

import optuna
from optkeras.optkeras import OptKeras

print(f'TensorFlow version : {tf.__version__}')
print(f'Keras version      : {keras.__version__}')
print(f'Optuna version     : {optuna.__version__}')
print(f'GPU available      : {tf.config.list_physical_devices("GPU")}')

## 3. Configuration
All hyperparameters and paths in one place for reproducibility.

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
# Update these paths to match your local environment or Google Drive mount
INFECTED_PATH   = '/content/drive/MyDrive/dataset/Parasitized'
UNINFECTED_PATH = '/content/drive/MyDrive/dataset/Uninfected'
MODEL_SAVE_PATH = '/content/drive/MyDrive/models/'
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)

# ── Image settings ─────────────────────────────────────────────────────────
IMG_SIZE    = (224, 224)   # Input resolution for CNN
IMG_SHAPE   = (224, 224, 3)
NUM_CLASSES = 1            # Binary output (sigmoid)

# ── Training settings ──────────────────────────────────────────────────────
EPOCHS      = 150
BATCH_SIZE  = 32
K_FOLDS     = 3
LEARNING_RATE = 1e-4       # Best value found via Optuna
DROPOUT_RATE  = 0.2

# ── Labels ─────────────────────────────────────────────────────────────────
LABEL_PARASITIZED = 1
LABEL_UNINFECTED  = 0
CLASS_NAMES = ['Uninfected', 'Parasitized']

print('Configuration loaded.')

## 4. Data Loading & Preprocessing

**Step 1 of pipeline:** Dataset → Data Preprocessing

Images are loaded, resized to 224×224 (required for CNN input), and converted to numpy arrays.
Corrupted or unreadable files are caught and skipped via try/except.

In [ ]:
def preprocess_data(file_list, label):
    """
    Load and preprocess a list of image files.

    Args:
        file_list (list): Full paths to image files.
        label (int): Class label — 1 for parasitized, 0 for uninfected.

    Returns:
        data   (list): Numpy arrays of shape (224, 224, 3).
        labels (list): Corresponding integer labels.

    Design choice: images are resized to 224x224 to match the receptive field
    expected by the CNN architecture and to remain compatible with VGG-family
    models used in comparative experiments.
    """
    data, labels = [], []
    for file in file_list:
        try:
            image = tf.keras.preprocessing.image.load_img(
                file, target_size=IMG_SIZE
            )
            image_array = tf.keras.preprocessing.image.img_to_array(image)
            data.append(image_array)
            labels.append(label)
        except Exception as e:
            print(f'[WARNING] Skipping unreadable file {file}: {e}')
    return data, labels


# ── Load infected images ───────────────────────────────────────────────────
infected_files   = [
    os.path.join(INFECTED_PATH, f)
    for f in os.listdir(INFECTED_PATH)
    if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))
]
uninfected_files = [
    os.path.join(UNINFECTED_PATH, f)
    for f in os.listdir(UNINFECTED_PATH)
    if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))
]

print(f'Found {len(infected_files):,} parasitized images')
print(f'Found {len(uninfected_files):,} uninfected images')

# ── Preprocess ────────────────────────────────────────────────────────────
infected_data,   infected_labels   = preprocess_data(infected_files,   LABEL_PARASITIZED)
uninfected_data, uninfected_labels = preprocess_data(uninfected_files, LABEL_UNINFECTED)

# ── Combine & convert ─────────────────────────────────────────────────────
data   = np.array(infected_data   + uninfected_data,   dtype='float32')
labels = np.array(infected_labels + uninfected_labels, dtype='float32')

print(f'\nCombined dataset shape : {data.shape}')
print(f'Labels shape           : {labels.shape}')
print(f'Class balance          : {np.sum(labels==1):,} parasitized | {np.sum(labels==0):,} uninfected')

## 5. Data Augmentation

**Step 2 of pipeline:** Data Augmentation

Augmentation simulates the variability observed across different microscopes, staining protocols,
and imaging conditions — particularly important given the distribution shift between the NIH dataset
(standardised lab conditions) and our locally collected Algerian images (7 cities, variable protocols).

- `rescale=1./255` — normalises pixel values to [0, 1] for gradient stability  
- `rotation_range=40` — simulates slide rotation under microscope  
- `zoom_range=0.2` — simulates different magnification levels  
- `horizontal_flip=True` — cells have no fixed orientation

In [ ]:
datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale            = 1./255,
    rotation_range     = 40,
    width_shift_range  = 0.2,
    height_shift_range = 0.2,
    shear_range        = 0.2,
    zoom_range         = 0.2,
    horizontal_flip    = True,
    fill_mode          = 'nearest'
)

# Validation generator — only rescales, no augmentation on val data
val_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale = 1./255
)

# ── Visualise augmentation examples ───────────────────────────────────────
sample_img = data[0:1]  # Take one sample image
aug_iter   = datagen.flow(sample_img, batch_size=1)

fig, axes = plt.subplots(1, 6, figsize=(16, 3))
axes[0].imshow(sample_img[0] / 255.)
axes[0].set_title('Original', fontsize=9)
axes[0].axis('off')
for i in range(1, 6):
    aug_img = next(aug_iter)[0]
    axes[i].imshow(aug_img)
    axes[i].set_title(f'Augmented {i}', fontsize=9)
    axes[i].axis('off')
plt.suptitle('Data Augmentation Examples', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('augmentation_examples.png', dpi=150, bbox_inches='tight')
plt.show()
print('Augmentation pipeline ready.')

## 6. K-Fold Cross-Validation Setup

**K = 3 folds** — each fold uses ~67% for training and ~33% for validation.
Fold splits are deterministic (fixed via np.arange) ensuring consistent evaluation across runs.

In [ ]:
def kfold_indices(data, k):
    """
    Generate deterministic K-Fold train/validation index splits.

    Args:
        data (np.ndarray): Full dataset.
        k (int): Number of folds.

    Returns:
        folds (list of tuples): Each tuple is (train_indices, val_indices).

    Design choice: custom implementation rather than sklearn.KFold to maintain
    full control over the split logic and avoid implicit shuffling.
    """
    fold_size = len(data) // k
    indices   = np.arange(len(data))
    folds     = []
    for i in range(k):
        val_indices   = indices[i * fold_size : (i + 1) * fold_size]
        train_indices = np.concatenate([
            indices[:i * fold_size],
            indices[(i + 1) * fold_size:]
        ])
        folds.append((train_indices, val_indices))
    return folds


fold_indices = kfold_indices(data, K_FOLDS)

for i, (train_idx, val_idx) in enumerate(fold_indices):
    print(f'Fold {i+1}: {len(train_idx):,} train | {len(val_idx):,} val')

## 7. Custom CNN Architecture

**Steps 3–4 of pipeline:** CNN Initialisation → Feature Extraction

Architecture summary:
```
Input (224×224×3)
  → Conv2D(32, 3×3, ReLU) → MaxPool(2×2)
  → Conv2D(64, 3×3, ReLU) → MaxPool(2×2)
  → Conv2D(128, 3×3, ReLU) → MaxPool(2×2)
  → Flatten
  → Dense(128, ReLU)
  → Dropout(0.2)           ← regularisation to prevent overfitting
  → Dense(1, Sigmoid)      ← binary output: P(parasitized)
```

**Why custom CNN over VGG16/VGG19?**
Transfer learning models pre-trained on ImageNet encode features from natural images
(objects, textures) that differ significantly from microscopic blood-cell morphology.
Our custom CNN learns domain-specific low-level filters directly from the malaria images,
converging faster and achieving higher accuracy on this specialised task.

In [ ]:
def build_custom_cnn(input_shape=IMG_SHAPE, dropout_rate=DROPOUT_RATE, learning_rate=LEARNING_RATE):
    """
    Build and compile the custom CNN for malaria binary classification.

    Args:
        input_shape   (tuple): Image dimensions (H, W, C).
        dropout_rate  (float): Dropout probability after Dense layer.
        learning_rate (float): Adam optimiser learning rate.

    Returns:
        model (keras.Sequential): Compiled Keras model.

    Loss: Binary Cross-Entropy
        L(y, ŷ) = -[y·log(ŷ) + (1-y)·log(1-ŷ)]
    Regularisation: Dropout(0.2) — zeroes neurons randomly during training
        to prevent co-adaptation and reduce overfitting.
    """
    model = Sequential([
        # ── Block 1: low-level feature detection ──────────────────────────
        Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=input_shape,
               name='conv1'),
        MaxPooling2D((2, 2), name='pool1'),

        # ── Block 2: mid-level feature extraction ─────────────────────────
        Conv2D(64, (3, 3), activation='relu', padding='same', name='conv2'),
        MaxPooling2D((2, 2), name='pool2'),

        # ── Block 3: high-level pattern recognition ───────────────────────
        Conv2D(128, (3, 3), activation='relu', padding='same', name='conv3'),
        MaxPooling2D((2, 2), name='pool3'),

        # ── Classifier head ───────────────────────────────────────────────
        Flatten(name='flatten'),
        Dense(128, activation='relu', name='fc1'),
        Dropout(dropout_rate, name='dropout'),   # Key regularisation step
        Dense(1, activation='sigmoid', name='output')  # P(parasitized)
    ], name='malaria_cnn')

    model.compile(
        optimizer = Adam(learning_rate=learning_rate),
        loss      = 'binary_crossentropy',
        metrics   = ['accuracy']
    )
    return model


# Preview architecture
preview_model = build_custom_cnn()
preview_model.summary()

## 8. Hyperparameter Tuning with Optuna

**Step 5 of pipeline:** Fine Tuning

Optuna searches for the optimal learning rate using the optkeras wrapper.
Search space: learning rate in [1e-5, 1e-3] (log scale).  
Best value found: **lr = 1e-4** with Adam optimiser.

In [ ]:
# ── Quick Optuna search on fold 0 ─────────────────────────────────────────
# Uses a short run (10 epochs, 5 trials) to find best lr before full training

train_idx_0, val_idx_0 = fold_indices[0]
x_train_0 = data[train_idx_0] / 255.0
y_train_0 = labels[train_idx_0]
x_val_0   = data[val_idx_0]   / 255.0
y_val_0   = labels[val_idx_0]

study_name = 'malaria_cnn_lr_search'
ok = OptKeras(study_name=study_name)

def objective(trial):
    """
    Optuna objective: search over learning rate.
    Each trial builds a fresh model to avoid weight leakage between trials.
    """
    K.clear_session()  # Reset TF graph between trials

    lr = trial.suggest_float('learning_rate', 1e-5, 1e-3, log=True)

    model = Sequential([
        Conv2D(32,  (3,3), activation='relu', padding='same', input_shape=IMG_SHAPE),
        MaxPooling2D((2,2)),
        Conv2D(64,  (3,3), activation='relu', padding='same'),
        MaxPooling2D((2,2)),
        Conv2D(128, (3,3), activation='relu', padding='same'),
        MaxPooling2D((2,2)),
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(DROPOUT_RATE),
        Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer = Adam(learning_rate=lr),
        loss      = 'binary_crossentropy',
        metrics   = ['accuracy']
    )
    model.fit(
        x_train_0, y_train_0,
        validation_data = (x_val_0, y_val_0),
        batch_size  = BATCH_SIZE,
        epochs      = 10,           # Short run for search phase
        callbacks   = ok.callbacks(trial),
        verbose     = 0
    )
    return ok.trial_best_value

# Run optimisation (timeout=120s for demo; increase for real search)
ok.optimize(objective, n_trials=5, timeout=120)

best_lr = ok.study.best_params['learning_rate']
print(f'\nBest learning rate found: {best_lr:.6f}')
print(f'Best val accuracy       : {ok.study.best_value:.4f}')

## 9. Full K-Fold Training

**Steps 5–6 of pipeline:** Training → Fine Tuning

Training runs for 150 epochs across K=3 folds using the best learning rate found by Optuna.
The best model checkpoint per fold is saved to disk.

In [ ]:
fold_histories  = []
fold_val_acc    = []
best_model_path = None
best_val_acc    = 0.0

for fold_num, (train_idx, val_idx) in enumerate(fold_indices):
    print(f'\n{'='*60}')
    print(f'  FOLD {fold_num + 1} / {K_FOLDS}')
    print(f'{'='*60}')

    # ── Prepare fold data ──────────────────────────────────────────────────
    x_train = data[train_idx] / 255.0
    y_train = labels[train_idx]
    x_val   = data[val_idx]   / 255.0
    y_val   = labels[val_idx]

    print(f'  Train: {x_train.shape[0]:,} | Val: {x_val.shape[0]:,}')

    # ── Build fresh model for this fold ───────────────────────────────────
    K.clear_session()
    model = build_custom_cnn(learning_rate=best_lr)

    # ── Callbacks ─────────────────────────────────────────────────────────
    checkpoint_path = os.path.join(
        MODEL_SAVE_PATH, f'cnn_malaria_best_fold{fold_num+1}.h5'
    )
    callbacks = [
        ModelCheckpoint(
            filepath         = checkpoint_path,
            monitor          = 'val_accuracy',
            save_best_only   = True,
            mode             = 'max',
            verbose          = 1
        )
    ]

    # ── Train ──────────────────────────────────────────────────────────────
    train_gen = datagen.flow(x_train, y_train, batch_size=BATCH_SIZE)

    history = model.fit(
        train_gen,
        validation_data = (x_val, y_val),
        epochs          = EPOCHS,
        callbacks       = callbacks,
        verbose         = 1
    )

    fold_histories.append(history)
    final_val_acc = max(history.history['val_accuracy'])
    fold_val_acc.append(final_val_acc)

    print(f'\n  Fold {fold_num+1} best val_accuracy: {final_val_acc:.4f}')

    if final_val_acc > best_val_acc:
        best_val_acc    = final_val_acc
        best_model_path = checkpoint_path

print(f'\n{"="*60}')
print(f'K-FOLD RESULTS SUMMARY')
print(f'{"="*60}')
for i, acc in enumerate(fold_val_acc):
    print(f'  Fold {i+1}: val_accuracy = {acc:.4f}')
print(f'  Mean accuracy : {np.mean(fold_val_acc):.4f} ± {np.std(fold_val_acc):.4f}')
print(f'  Best checkpoint: {best_model_path}')

## 10. Training Curves

Plot loss and accuracy curves for all folds to inspect convergence and detect overfitting.
**No overfitting observed**: training and validation curves remain closely aligned across all 150 epochs.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

for fold_num, history in enumerate(fold_histories):
    c = colors[fold_num]
    axes[0].plot(history.history['loss'],     color=c, linestyle='-',  alpha=0.8, label=f'Fold {fold_num+1} Train')
    axes[0].plot(history.history['val_loss'], color=c, linestyle='--', alpha=0.8, label=f'Fold {fold_num+1} Val')
    axes[1].plot(history.history['accuracy'],     color=c, linestyle='-',  alpha=0.8, label=f'Fold {fold_num+1} Train')
    axes[1].plot(history.history['val_accuracy'], color=c, linestyle='--', alpha=0.8, label=f'Fold {fold_num+1} Val')

axes[0].set_title('Training and Validation Loss',     fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

axes[1].set_title('Training and Validation Accuracy', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves_kfold.png', dpi=150, bbox_inches='tight')
plt.show()
print('Training curves saved.')

## 11. Evaluation & Error Analysis

**Step 7 of pipeline:** Parasite Identification → Performance Metrics Computing

Load best checkpoint and evaluate on the validation fold.
Error analysis includes confusion matrix and misclassified sample inspection.

In [ ]:
from keras.models import load_model

# ── Load best model ────────────────────────────────────────────────────────
best_model = load_model(best_model_path)
print(f'Loaded checkpoint: {best_model_path}')

# ── Evaluate on best fold's validation set ────────────────────────────────
best_fold_idx  = np.argmax(fold_val_acc)
_, val_idx_best = fold_indices[best_fold_idx]
x_val_best = data[val_idx_best] / 255.0
y_val_best = labels[val_idx_best]

y_pred_prob = best_model.predict(x_val_best, verbose=0).flatten()
y_pred      = (y_pred_prob >= 0.5).astype(int)

acc     = accuracy_score(y_val_best, y_pred)
auc     = roc_auc_score(y_val_best, y_pred_prob)

print(f'\n── Final Validation Metrics (Best Fold) ──────────────────')
print(f'  Accuracy  : {acc:.4f} ({acc*100:.2f}%)')
print(f'  AUC-ROC   : {auc:.4f}')
print(f'\n── Classification Report ─────────────────────────────────')
print(classification_report(y_val_best, y_pred, target_names=CLASS_NAMES))

In [ ]:
# ── Confusion Matrix ───────────────────────────────────────────────────────
cm = confusion_matrix(y_val_best, y_pred)
tn, fp, fn, tp = cm.ravel()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
    linewidths=0.5, ax=ax
)
ax.set_title('Confusion Matrix — Best Fold', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Predicted Label', fontsize=11)
ax.set_ylabel('True Label',      fontsize=11)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n── Error Analysis ────────────────────────────────────────')
print(f'  True Positives  (TP): {tp:,}  — correctly detected parasitized')
print(f'  True Negatives  (TN): {tn:,}  — correctly detected uninfected')
print(f'  False Positives (FP): {fp:,}  — uninfected flagged as parasitized')
print(f'  False Negatives (FN): {fn:,}  — parasitized missed (most critical)')
print(f'\n  Sensitivity (Recall): {tp/(tp+fn):.4f}  ← clinically critical')
print(f'  Specificity         : {tn/(tn+fp):.4f}')
print(f'\n  Note: False negatives are the dominant failure mode on local')
print(f'  Algerian images due to variable staining across 7 collection cities.')

## 12. Misclassified Sample Inspection

Visualise examples the model got wrong to understand failure modes.

In [ ]:
# ── Find misclassified samples ─────────────────────────────────────────────
misclassified_idx = np.where(y_pred != y_val_best)[0]
print(f'Total misclassified: {len(misclassified_idx)} / {len(y_val_best)}')

n_show = min(8, len(misclassified_idx))
if n_show > 0:
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    axes = axes.flatten()
    for i, idx in enumerate(misclassified_idx[:n_show]):
        axes[i].imshow(x_val_best[idx])
        true_label = CLASS_NAMES[int(y_val_best[idx])]
        pred_label = CLASS_NAMES[int(y_pred[idx])]
        conf       = y_pred_prob[idx] if y_pred[idx] == 1 else 1 - y_pred_prob[idx]
        axes[i].set_title(
            f'True: {true_label}\nPred: {pred_label} ({conf:.2f})',
            fontsize=8, color='red'
        )
        axes[i].axis('off')
    for j in range(i+1, len(axes)):
        axes[j].axis('off')
    plt.suptitle('Misclassified Examples — Error Analysis', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('misclassified_examples.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No misclassified samples found in this fold — perfect validation score!')

## 13. Results Summary

Final consolidated results across all folds.

In [ ]:
print('=' * 60)
print('  MALARIA CNN — FINAL RESULTS SUMMARY')
print('=' * 60)
print(f'  Model        : Custom CNN (3 Conv blocks + Dropout(0.2))')
print(f'  Dataset      : {len(data):,} images (14,000 parasitized + 14,000 uninfected)')
print(f'  Sources      : NIH/LHNCBC (27,558) + Local Algeria (442)')
print(f'  Validation   : {K_FOLDS}-Fold Cross-Validation')
print(f'  Epochs       : {EPOCHS}')
print(f'  Best LR      : {best_lr:.2e}  (found via Optuna)')
print(f'  Dropout      : {DROPOUT_RATE}')
print(f'  Loss fn      : Binary Cross-Entropy')
print()
print(f'  Per-fold val accuracy:')
for i, acc in enumerate(fold_val_acc):
    marker = ' ← best' if i == best_fold_idx else ''
    print(f'    Fold {i+1}: {acc:.4f} ({acc*100:.2f}%){marker}')
print()
print(f'  Mean val accuracy : {np.mean(fold_val_acc):.4f} ± {np.std(fold_val_acc):.4f}')
print(f'  Best checkpoint   : {os.path.basename(best_model_path)}')
print('=' * 60)
print()
print('Publications:')
print('  • Baissi et al. (2024). Paludism Diagnosis Using Deep Learning.')
print('    ICEIS 2024, Springer. DOI: 10.2991/978-94-6463-496-9_19')
print('  • Baissi & Ammari (2024). Paludism Detection using CNN.')
print('    ISNIB 2024, IEEE. DOI: 10.1109/ISNIB64820.2025.10982943')